In [1]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import networkx as nx
import itertools
import math
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
from same_decision_probability_calculation import *
from minimum_information_loss_partition import *
from utils import *

from monte_carlo_sdp import *

In [3]:
from pgmpy.utils import get_example_model

# Loading Models

In [4]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from pgmpy.models import NaiveBayes
from pgmpy.estimators import MaximumLikelihoodEstimator

# ── VOTING ──────────────────────────────────────────────────────────────────
voting = fetch_ucirepo(id=105)
df_voting = pd.concat([voting.data.features, voting.data.targets], axis=1)
df_voting.columns = [c.strip() for c in df_voting.columns]

# Replace '?' missing values — Naive Bayes needs complete data
df_voting = df_voting.replace('?', pd.NA).dropna()

# All values must be strings/categories for pgmpy
df_voting = df_voting.astype(str)

target_voting = 'Class'   # 'democrat' / 'republican'

voting_model = NaiveBayes()
voting_model.fit(df_voting, target_voting,
                 estimator=MaximumLikelihoodEstimator)

# ── CHESS ────────────────────────────────────────────────────────────────────
chess = fetch_ucirepo(id=22)
df_chess = pd.concat([chess.data.features, chess.data.targets], axis=1)
df_chess = df_chess.astype(str)

target_chess = 'skach' 

chess_model = NaiveBayes()
chess_model.fit(df_chess, target_chess,
                estimator=MaximumLikelihoodEstimator)

In [5]:
df_chess.skach.value_counts()

t    2216
f     980
Name: skach, dtype: int64

In [6]:
# cast models to pgmpy BayesianNetwork for compatibility with our code
voting_model = BayesianNetwork(voting_model.edges())
chess_model = BayesianNetwork(chess_model.edges())

# fit
voting_model.fit(df_voting, estimator=MaximumLikelihoodEstimator)
chess_model.fit(df_chess, estimator=MaximumLikelihoodEstimator)

In [7]:
# find binary variables in chess df
binary_vars_chess = [col for col in df_chess.columns if df_chess[col].nunique() == 2]
print(f"Binary variables in Chess dataset: {binary_vars_chess}")

Binary variables in Chess dataset: ['bkblk', 'bknwy', 'bkon8', 'bkona', 'bkspr', 'bkxbq', 'bkxcr', 'bkxwp', 'blxwp', 'bxqsq', 'cntxt', 'dsopp', 'dwipd', 'katri', 'mulch', 'qxmsq', 'r2ar8', 'reskd', 'reskr', 'rimmx', 'rkxwp', 'rxmsq', 'simpl', 'skach', 'skewr', 'skrxp', 'spcop', 'stlmt', 'thrsk', 'wkcti', 'wkna8', 'wknck', 'wkovl', 'wkpos', 'wtoeg']


In [8]:
sdp_voting = naive_bayes_sdp(
    model=voting_model,
    D=target_voting,
    d_value='democrat',
    evidence={},
    threshold=0.5,
)

In [9]:
sdp_voting_fast = fast_broadcast_sdp(
    model=voting_model,
    D=target_voting,
    d_value='republican',
    evidence={},
    threshold=0.5,
    partitions= get_partitions(voting_model, voting_model.nodes(), target_voting, {})
)

In [10]:
sdp_voting

0.5343369815465464

In [11]:
sdp_voting_fast


0.5343369815465464

In [8]:
alarm_model = get_example_model('alarm')
child_model = get_example_model('child')
#asia_model = get_example_model('asia')
insurance_model = get_example_model('insurance')
hailfinder_model = get_example_model('hailfinder')
hepar_model = get_example_model('hepar2')
barley_model = get_example_model('barley')
win95pts_model = get_example_model('win95pts')
#mildew_model = get_example_model('mildew')
#water_model = get_example_model('water')
mildew_model = None
water_model = None

In [9]:
child_model.name = 'child'
insurance_model.name = 'insurance'
alarm_model.name = 'alarm'
hepar_model.name = 'hepar'
hailfinder_model.name = 'hailfinder'
win95pts_model.name = 'win95pts'
barley_model.name = 'barley'
voting_model.name = 'voting'
chess_model.name = 'chess'
#mildew_model.name = 'mildew'
#water_model.name = 'water'

In [14]:
# cardinality of variables
for node in insurance_model.nodes():
    print(f"Cardinality of variable '{node}': {insurance_model.get_cardinality(node)}")

Cardinality of variable 'GoodStudent': 2
Cardinality of variable 'Age': 3
Cardinality of variable 'SocioEcon': 4
Cardinality of variable 'RiskAversion': 4
Cardinality of variable 'VehicleYear': 2
Cardinality of variable 'ThisCarDam': 4
Cardinality of variable 'RuggedAuto': 3
Cardinality of variable 'Accident': 4
Cardinality of variable 'MakeModel': 5
Cardinality of variable 'DrivQuality': 3
Cardinality of variable 'Mileage': 4
Cardinality of variable 'Antilock': 2
Cardinality of variable 'DrivingSkill': 3
Cardinality of variable 'SeniorTrain': 2
Cardinality of variable 'ThisCarCost': 4
Cardinality of variable 'Theft': 2
Cardinality of variable 'CarValue': 5
Cardinality of variable 'HomeBase': 4
Cardinality of variable 'AntiTheft': 2
Cardinality of variable 'PropCost': 4
Cardinality of variable 'OtherCarCost': 4
Cardinality of variable 'OtherCar': 2
Cardinality of variable 'MedCost': 4
Cardinality of variable 'Cushioning': 4
Cardinality of variable 'Airbag': 2
Cardinality of variable 'I

# Monte Carlo Single Tests

# Run Experiment

In [10]:
def get_target(model):
    targets = {
        'child': 'Sick',
        'alarm': 'HYPOVOLEMIA',
        'barley': 'pesticid',
        'insurance': 'Theft',
        'mildew': None, # no binary variables
        'water': None, # no binary variables
        'hailfinder': 'ScenRelAMCIN',
        'hepar': 'hepatomegaly',
        'win95pts': 'PrtMem',
        'voting': 'Class',
        'chess': 'skach'
    }
    # define the target manually when avaiable (from the respective paper) or randomly
    # conferir se vão ser esses mesmos!!

    return targets[model.name]

def get_h_ratio(model):
    ratios = {
        'child': 0.5,
        'alarm': 0.30,
        'hepar': 0.20,
        'barley': 0.20,
        'mildew': 0.30,
        'water': 0.30,
        'hailfinder': 0.30,
        'win95pts': 0.20,
        'insurance': 0.40,
        'voting': 0.5,
        'chess': 0.9 #era 0.86
    }
    return ratios[model.name]
    



In [12]:
len(chess_model.nodes())

36

In [11]:
models_to_run = [voting_model, chess_model, child_model, alarm_model, insurance_model, hailfinder_model, hepar_model, win95pts_model]

In [ ]:
#models_to_run = [chess_model]

In [36]:
for model in models_to_run:
    print(f"Model '{model.name}' has {len(model.nodes())} variables.")

Model 'chess' has 36 variables.


In [37]:
len(models_to_run)

1

In [14]:
all_targets_are_binary = True
for bn in models_to_run:
    #print(f"\n=== BN: {bn.name} ===")
    target = get_target(bn)
    if target is None:
        #print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    if len(target_states) != 2:
        #print(f"--> Target '{target}' in {bn.name} is not binary (States: {target_states}), skipping.")
        all_targets_are_binary = False
        continue
    #print(f"Available states for target '{target}': {target_states}")
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    #print(f"Target Node: {target}, Target Value: {target_value}")

print(f"\nAll targets are binary: {all_targets_are_binary}")


All targets are binary: True


In [15]:
for bn in models_to_run:
    print(f"\n=== BN: {bn.name} ===")
    all_nodes = list(bn.nodes())
    
    target = get_target(bn)
    if target is None:
        print(f"--> No binary target defined for {bn.name}, skipping.")
        continue
    target_states = bn.get_cpds(target).state_names[target]
    target_value = target_states[1] if len(target_states) > 1 else target_states[0]
    print(f"Target Node: {target}, Target Value: {target_value}")
    
    available_nodes = [n for n in all_nodes if n != target]
    print(f"H ratio: {get_h_ratio(bn)}")
    n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
    print(f"using {n_hidden} H variables")


=== BN: voting ===
Target Node: Class, Target Value: republican
H ratio: 0.5
using 8 H variables

=== BN: chess ===
Target Node: skach, Target Value: t
H ratio: 0.9
using 31 H variables

=== BN: child ===
Target Node: Sick, Target Value: no
H ratio: 0.5
using 9 H variables

=== BN: alarm ===
Target Node: HYPOVOLEMIA, Target Value: FALSE
H ratio: 0.3
using 10 H variables

=== BN: insurance ===
Target Node: Theft, Target Value: False
H ratio: 0.4
using 10 H variables

=== BN: hailfinder ===
Target Node: ScenRelAMCIN, Target Value: CThruK
H ratio: 0.3
using 16 H variables

=== BN: hepar ===
Target Node: hepatomegaly, Target Value: absent
H ratio: 0.2
using 13 H variables

=== BN: win95pts ===
Target Node: PrtMem, Target Value: Less_than_2Mb
H ratio: 0.2
using 15 H variables


In [ ]:
import time
from xml.parsers.expat import model
def run_targeted_sdp_experiment(output_csv="targeted_sdp_benchmark.csv"):
    
    results = []
    raw_results = []
    #H_RATIO = 0.20
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.40, 0.50, 0.6, 0.70, 0.8, 0.9, 1.0]
    MCMC_TRIALS = 10 
    
    for bn in models_to_run:
        n_nodes = bn.number_of_nodes()
        print(f"\n========================================")
        print(f"Processing BN: {bn.name}")
        
        all_nodes = list(bn.nodes())
        
        target = get_target(bn)
        if target is None:
            print(f"--> No binary target defined for {bn.name}, skipping.")
            continue
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        print(f"Target Node: {target}, Target Value: {target_value}")
        
        available_nodes = [n for n in all_nodes if n != target]
        print(f"H ratio: {get_h_ratio(bn)}")
        n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
        print(f"using {n_hidden} H variables")
        n_evidence = len(available_nodes) - n_hidden
        print(f"and {n_evidence} evidence variables")
        #hidden_vars = random.sample(available_nodes, n_hidden)
        #evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
        # Run the Harvester 
        #harvested_data = harvest_patients_for_all_buckets(
        #    bn, target, target_value, DECISION_THRESHOLD, evidence_vars, TARGET_BUCKETS
        #)
        if bn.name == 'ignore':
            # use only 0.5 bucket
            harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                          n_evidence, buckets=[0.5])
        else:
            harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                                  n_evidence, buckets=TARGET_BUCKETS)
            
        # try to fill remaining buckets with harvest_patients_for_all_buckets
        #empty_buckets = [b for b, v in harvested_data.items() if v is None]
        #print(f"Trying to fill remaining buckets with harvest_patients_for_all_buckets...")
        #if empty_buckets:
        #    additional_data = harvest_patients_for_all_buckets(
        #        bn, target, target_value, DECISION_THRESHOLD, evidence_vars, empty_buckets
        #    )
        #    harvested_data.update(additional_data)
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            hidden_vars = [n for n in bn.nodes() if n not in patient and n != target]
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # RACE TIMING 1: EXACT SDP
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            exact_time = np.nan
            
            try:
                start_time = time.time()
                # Re-run the exact calculation once just to time it cleanly
                exact_sdp_benchmark = fast_broadcast_sdp(bn, target, target_value, patient, DECISION_THRESHOLD, partitions)
                exact_time = time.time() - start_time
                print(f"       -> Exact Time: {exact_time:.4f} seconds")
            except (ValueError, MemoryError):
                print(f"       -> Exact Time: [FAILED DUE TO MEMORY/EINSUM LIMIT]")
            
            # ========================================================
            # RACE TIMING 2: MCMC SDP
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            
            for trial in range(MCMC_TRIALS):
                start_time = time.time()
                est_sdp = fast_mcmc_sdp_estimation(
                    bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=7000, burn_in=500, thinning=5
                )
                mcmc_times.append(time.time() - start_time)
                mcmc_estimates.append(est_sdp)
                raw_results.append({
                    'Network': bn.name,
                    'Target_Bucket': target_sdp,
                    'Exact_SDP': exact_sdp,
                    'MCMC_Estimate': est_sdp,
                    'Exact_Time_sec': exact_time, 
                    'MCMC_Time_sec': mcmc_times[-1]
                })
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_variance = np.var(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            
            print(f"       -> MCMC Mean Estimate: {mcmc_mean:.4f}")
            print(f"       -> MCMC Avg Time: {mcmc_avg_time:.4f} seconds")
            
            absolute_error = abs(exact_sdp - mcmc_mean)
            
            # Record everything to the dataset
            results.append({
                'Network': bn.name,
                'N_Nodes': n_nodes,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)
            pd.DataFrame(raw_results).to_csv("raw_" + output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [77]:
# todo: compare the MCMC acceptance rate varying the size of H in the chess network (H = 14 and H = 30)

In [78]:
#run_targeted_sdp_experiment()

# Run Experiment Memory Monitor

In [48]:
import time
from xml.parsers.expat import model
import tracemalloc

models_to_run = [hepar_model, win95pts_model]

def run_with_profiler(func, *args, **kwargs):
    """
    Executes a function while tracking execution time and peak memory allocation.
    Returns: (Result, Time_in_seconds, Peak_Memory_in_MB)
    """
    # Start tracing memory allocations
    tracemalloc.start()
    start_time = time.time()
    
    # Run the target algorithm
    result = None
    try:
        result = func(*args, **kwargs)
        status = "Success"
    except MemoryError:
        status = "MemoryError"
    except ValueError as e:
        status = f"ValueError: {e}"
        
    # Stop the clock and memory tracer
    end_time = time.time()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    execution_time = end_time - start_time
    peak_memory_mb = peak_mem / (1024 * 1024)
    
    if status != "Success":
        raise MemoryError(f"Crashed at {peak_memory_mb:.2f} MB")
        
    return result, execution_time, peak_memory_mb

def select_random_target(bn):
    # list all binary variables in the BN
    binary_vars = []
    for node in bn.nodes():
        cpd = bn.get_cpds(node)
        if cpd is not None and len(cpd.state_names[node]) == 2:
            binary_vars.append(node)
    if not binary_vars:
        return None, None
    print(f"Binary variables in {bn.name}: {binary_vars}")
    selected_target = random.choice(binary_vars)
    return selected_target

def run_targeted_sdp_experiment_memory(output_csv="targeted_sdp_benchmark_memory.csv"):
    
    results = []
    raw_results = []
    #H_RATIO = 0.20
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.7, 0.8, 0.9, 1.0]
    MCMC_TRIALS = 3 
    
    for bn in models_to_run:
        n_nodes = bn.number_of_nodes()
        print(f"\n========================================")
        print(f"Processing BN: {bn.name}")
        
        all_nodes = list(bn.nodes())
        
        target = get_target(bn)
        #target = select_random_target(bn)
        if target is None:
            print(f"--> No binary target defined for {bn.name}, skipping.")
            continue
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        print(f"Target Node: {target}, Target Value: {target_value}")
        
        available_nodes = [n for n in all_nodes if n != target]
        print(f"H ratio: {get_h_ratio(bn)}")
        #n_hidden = max(1, int(len(available_nodes) * get_h_ratio(bn)))
        n_hidden = 22
        if bn.name == 'insurance':
            n_hidden = 11
        print(f"using {n_hidden} H variables")
        n_evidence = len(available_nodes) - n_hidden
        print(f"and {n_evidence} evidence variables")
        #hidden_vars = random.sample(available_nodes, n_hidden)
        #evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
        if bn.name == 'hailfinder':
            
            harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                          n_evidence, buckets=[1.0])
        else:
            all_nodes = list(bn.nodes())
            available_nodes = [n for n in all_nodes if n != target]
            # pick random n_evidence variables to lock as evidence
            evidence_vars = random.sample(available_nodes, min(n_evidence, len(available_nodes)))
            harvested_data = find_exact_experimental_patients_random(bn, target, target_value, DECISION_THRESHOLD,
                                                                  n_evidence, buckets=TARGET_BUCKETS)
            
        # try to fill remaining buckets with harvest_patients_for_all_buckets
        #empty_buckets = [b for b, v in harvested_data.items() if v is None]
        #print(f"Trying to fill remaining buckets with harvest_patients_for_all_buckets...")
        #if empty_buckets:
        #    additional_data = harvest_patients_for_all_buckets(
        #        bn, target, target_value, DECISION_THRESHOLD, evidence_vars, empty_buckets
        #    )
        #    harvested_data.update(additional_data)
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            hidden_vars = [n for n in bn.nodes() if n not in patient and n != target]
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # RACE TIMING 1: EXACT SDP
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            exact_time = np.nan
            exact_mem_mb = np.nan
            
            print(f"       -> Running Exact SDP...")
            try:
                exact_sdp, exact_time, exact_mem_mb = run_with_profiler(
                    fast_broadcast_sdp, bn, target, target_value, patient, DECISION_THRESHOLD, partitions
                )
                print(f"          Time: {exact_time:.4f} sec | Peak Memory: {exact_mem_mb:.2f} MB")
            except Exception as e:
                print(f"          [FAILED]: {e}")
            
            # ========================================================
            # RACE TIMING 2: MCMC SDP
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            mcmc_memories = []
            
            print(f"       -> Running MCMC SDP (Trials: {MCMC_TRIALS})...")
            for trial in range(MCMC_TRIALS):
                est_sdp, t_time, t_mem = run_with_profiler(
                    fast_mcmc_sdp_estimation, bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=1000, burn_in=50, thinning=5
                )
                mcmc_estimates.append(est_sdp)
                mcmc_times.append(t_time)
                mcmc_memories.append(t_mem)
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_variance = np.var(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            mcmc_avg_mem_mb = np.mean(mcmc_memories) # Average peak memory across trials
            
            print(f"          Avg Time: {mcmc_avg_time:.4f} sec | Peak Memory: {mcmc_avg_mem_mb:.2f} MB")
            
            absolute_error = abs(exact_sdp - mcmc_mean)
            
            # Record everything to the dataset
            results.append({
                'Network': bn.name,
                'N_Nodes': n_nodes,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error,
                'Exact_Peak_Memory_MB': exact_mem_mb,
                'MCMC_Avg_Peak_Memory_MB': mcmc_avg_mem_mb
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)
            pd.DataFrame(raw_results).to_csv("raw_" + output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [49]:
run_targeted_sdp_experiment_memory()


Processing BN: hepar
Target Node: hepatomegaly, Target Value: absent
H ratio: 0.2
using 22 H variables
and 47 evidence variables

Hunting for patients... (Locking 47 variables as evidence)
Generating batch 1/2 of 8000 random realities...
--> Filled bucket 1.0 with Exact SDP: 0.9993
--> Filled bucket 0.8 with Exact SDP: 0.8370
--> Filled bucket 0.9 with Exact SDP: 0.9387
--> Filled bucket 0.7 with Exact SDP: 0.7033
All buckets filled successfully!

  -> Benchmarking found patient for bucket 0.7 (Exact: 0.7033)
       -> Running Exact SDP...
          Time: 0.2945 sec | Peak Memory: 11.48 MB
       -> Running MCMC SDP (Trials: 3)...
          Avg Time: 2.1189 sec | Peak Memory: 2.66 MB

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.8370)
       -> Running Exact SDP...
          Time: 1.4708 sec | Peak Memory: 91.22 MB
       -> Running MCMC SDP (Trials: 3)...
          Avg Time: 2.3231 sec | Peak Memory: 2.52 MB

  -> Benchmarking found patient for bucket 0.9 (Exact: 0.9387)


,Network,N_Nodes,Target_Bucket,Target_Node,Target_Value,Exact_SDP,Exact_Time_sec,MCMC_Mean_SDP,MCMC_Variance,MCMC_Avg_Time_sec,Absolute_Error,Exact_Peak_Memory_MB,MCMC_Avg_Peak_Memory_MB
0,hepar,70,0.7,hepatomegaly,absent,0.020684,0.294456,0.029000,0.000114,2.118867,8.316345e-03,11.482932,2.663857
1,hepar,70,0.8,hepatomegaly,absent,0.251382,1.470806,0.219667,0.001836,2.323111,3.171553e-02,91.215646,2.515592
2,hepar,70,0.9,hepatomegaly,absent,0.690664,0.795141,0.740333,0.002012,2.238926,4.966955e-02,60.858445,2.315228
3,hepar,70,1.0,hepatomegaly,absent,0.858430,0.046061,0.852333,0.000964,2.177838,6.096836e-03,0.203357,2.415595
4,win95pts,76,1.0,PrtMem,Less_than_2Mb,1.000000,0.054127,1.000000,0.000000,4.070969,1.110223e-16,0.664580,1.524617
